In [1]:
# https://github.com/google/earthengine-community/blob/master/guides/linked/Earth_Engine_TensorFlow_DNN_from_scratch.ipynb

In [2]:
import tensorflow as tf
import os

train_dataset = tf.data.TFRecordDataset("training.tfrecord.gz", compression_type='GZIP')

C:\Users\zizzi\gee\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
# Get the first record to check.
raw_record = iter(train_dataset).next()
print('raw_record:')
print(raw_record)

# Get the raw record as a numpy array.
record = raw_record.numpy()
print('record:')
print(record)

# Decode a serialized tf.Example as a proto.
example_proto = tf.train.Example.FromString(record)
print('example_proto:')
print(example_proto)

raw_record:
tf.Tensor(b'\n\x99\x01\n\x0e\n\x02B2\x12\x08\x12\x06\n\x04\xb6\x10d=\n\x12\n\x06random\x12\x08\x12\x06\n\x04Y\x88\xc0>\n\x0e\n\x02B3\x12\x08\x12\x06\n\x04b\xa1\x96=\n\x0e\n\x02B4\x12\x08\x12\x06\n\x04\xcc\xee\xc9=\n\x0e\n\x02B8\x12\x08\x12\x06\n\x04\x96!N>\n\x16\n\nbiome_type\x12\x08\x12\x06\n\x04\x00\x00\x80A\n\x10\n\x04NDVI\x12\x08\x12\x06\n\x04Os\xab>\n\x19\n\x0csystem:index\x12\t\n\x07\n\x051_5_0', shape=(), dtype=string)
record:
b'\n\x99\x01\n\x0e\n\x02B2\x12\x08\x12\x06\n\x04\xb6\x10d=\n\x12\n\x06random\x12\x08\x12\x06\n\x04Y\x88\xc0>\n\x0e\n\x02B3\x12\x08\x12\x06\n\x04b\xa1\x96=\n\x0e\n\x02B4\x12\x08\x12\x06\n\x04\xcc\xee\xc9=\n\x0e\n\x02B8\x12\x08\x12\x06\n\x04\x96!N>\n\x16\n\nbiome_type\x12\x08\x12\x06\n\x04\x00\x00\x80A\n\x10\n\x04NDVI\x12\x08\x12\x06\n\x04Os\xab>\n\x19\n\x0csystem:index\x12\t\n\x07\n\x051_5_0'
example_proto:
features {
  feature {
    key: "B2"
    value {
      float_list {
        value: 0.05568
      }
    }
  }
  feature {
    key: "B3"
    v

In [4]:
from pprint import pprint

FEATURE_NAMES = ["B2", "B3", "B4", "B8", "NDVI", "biome_type"]

# List of fixed-length features, all of which are float32.
columns = [
  tf.io.FixedLenFeature(shape=(), dtype=tf.float32) for k in FEATURE_NAMES
]

# Dictionary with names as keys, features as values.
features_dict = dict(zip(FEATURE_NAMES, columns))

pprint(features_dict)

{'B2': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None),
 'B3': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None),
 'B4': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None),
 'B8': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None),
 'NDVI': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None),
 'biome_type': FixedLenFeature(shape=(), dtype=tf.float32, default_value=None)}


In [5]:
LABEL = 'biome_type'

def parse_tfrecord(example_proto):
  """The parsing function.

  Read a serialized example into the structure defined by featuresDict.

  Args:
    example_proto: a serialized Example.

  Returns:
    A tuple of the predictors dictionary and the label, cast to an `int32`.
  """
  parsed_features = tf.io.parse_example(example_proto, features_dict)
  labels = parsed_features.pop(LABEL)
  return parsed_features, tf.cast(labels, tf.int32)

# Map the function over the dataset.
parsed_dataset = train_dataset.map(parse_tfrecord, num_parallel_calls=5)

# Print the first parsed record to check.
pprint(iter(parsed_dataset).next())

({'B2': <tf.Tensor: shape=(), dtype=float32, numpy=0.05567999929189682>,
  'B3': <tf.Tensor: shape=(), dtype=float32, numpy=0.07355000078678131>,
  'B4': <tf.Tensor: shape=(), dtype=float32, numpy=0.09860000014305115>,
  'B8': <tf.Tensor: shape=(), dtype=float32, numpy=0.2012999951839447>,
  'NDVI': <tf.Tensor: shape=(), dtype=float32, numpy=0.3348641097545624>},
 <tf.Tensor: shape=(), dtype=int32, numpy=16>)


In [6]:
from tensorflow import keras
INPUTS = ["B2", "B3", "B4", "B8", "NDVI"]
# The base model takes a vector of inputs and returns a vector of outputs.
inputs = keras.Input(shape=(len(INPUTS),), name='input_array')
x = tf.keras.layers.Dense(64, activation=tf.nn.relu)(inputs)
x = tf.keras.layers.Dropout(0.1)(x)
x = tf.keras.layers.Dense(32, activation=tf.nn.softmax)(x)
model = tf.keras.Model(inputs=inputs, outputs=x)

# A Layer to add an NDVI feature and stack the input tensors.
class MyPreprocessing(keras.layers.Layer):
  def __init__(self, **kwargs):
    super(MyPreprocessing, self).__init__(**kwargs)

  def call(self, features_dict):
    #features_dict = add_NDVI(features_dict)
    return tf.stack([features_dict[b] for b in INPUTS], axis=1)

# A Model that wraps the base model with the preprocessing layer.
class MyModel(keras.Model):
  def __init__(self, preprocessing, backbone, **kwargs):
    super().__init__(**kwargs)
    self.preprocessing = preprocessing
    self.backbone = backbone

  def call(self, features_dict):
    x = self.preprocessing(features_dict)
    return self.backbone(x)

wrapped_model = MyModel(MyPreprocessing(), model)

In [7]:
wrapped_model

<MyModel name=my_model, built=False>

In [8]:
# Compile the model with the specified loss function.
wrapped_model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model to the training data.
wrapped_model.fit(x=parsed_dataset.batch(4), epochs=10)

Epoch 1/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 1ms/step - accuracy: 0.5651 - loss: 1.4947
Epoch 2/10
   94/14935 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1055 - loss: 2.1871

C:\Users\zizzi\gee\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.6603 - loss: 1.1423
Epoch 3/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7147 - loss: 0.9923
Epoch 4/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7437 - loss: 0.8942
Epoch 5/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7601 - loss: 0.8562
Epoch 6/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7731 - loss: 0.7985
Epoch 7/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7795 - loss: 0.7780
Epoch 8/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7925 - loss: 0.7240
Epoch 9/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.8077 - loss: 0.6848
Epoch 10/10
14935/14935 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.8154 - loss: 0.6694


In [9]:
test_dataset = (
  tf.data.TFRecordDataset("testing.tfrecord.gz", compression_type='GZIP')
    .map(parse_tfrecord, num_parallel_calls=5)
    .batch(1))

wrapped_model.evaluate(test_dataset)

25597/25597 ━━━━━━━━━━━━━━━━━━━━ 32s 1ms/step - accuracy: 0.2338 - loss: 3.4726


[3.4725890159606934, 0.233816459774971]